# Audit: portion-matcher vs nutrient-matcher

The gram conversion uses a *simple* matcher (word overlap + substring) against the Helsedirektoratet portion table.
The nutrient lookup uses the *strict* matcher (synonyms, pre-filters, cooking-state) against Matvaretabellen / USDA.

This notebook quantifies how often the two matchers pick different foods, and estimates how much
recipe weight is affected — useful for the limitations section of the thesis.

In [1]:
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loaders.portion_helsedir import (
    load_portion_table,
    match_food_item,
    canonicalize_unit_text,
)

portion_df = load_portion_table()
print(f'Portion table rows: {len(portion_df):,}')

Portion table rows: 752


In [2]:
MODELS = ['gpt54_mini', 'gpt54_nano', 'gemini_flash', 'gemini_flash_lite', 'grok_fast', 'grok4_fast']
DATA = PROJECT_ROOT / 'data'

dfs = []
for m in MODELS:
    d = pd.read_csv(DATA / f'{m}_recipes_debug.csv')
    d['model'] = m
    dfs.append(d)

debug = pd.concat(dfs, ignore_index=True)
print(f'Total ingredient lines across {len(MODELS)} models: {len(debug):,}')

Total ingredient lines across 6 models: 78,984


In [3]:
# Filter to lines where the portion matcher actually runs:
#   - not ignored (salt/spices/water are excluded)
#   - unit is not g/kg (those skip matching)
#   - ingredient name is present

GRAM_UNITS = {'g', 'gram', 'grams', 'kg', 'kilogram', 'kilograms'}

ignored = debug['ignored'].fillna(False).astype(bool)
unit_lower = debug['unit'].astype(str).str.strip().str.lower()

non_gram = debug[(~ignored) & (~unit_lower.isin(GRAM_UNITS)) & debug['ingredient_name'].notna()].copy()
non_gram['unit_lower'] = non_gram['unit'].astype(str).str.strip().str.lower()

print(f'Non-gram ingredient lines (portion matcher runs here): {len(non_gram):,}')
print(f'Share of all lines: {len(non_gram) / len(debug):.1%}')
print('\nUnit distribution:')
print(non_gram['unit_lower'].value_counts().head(15))

Non-gram ingredient lines (portion matcher runs here): 23,890
Share of all lines: 30.2%

Unit distribution:
unit_lower
dl        6164
tbsp      5386
piece     4098
cloves    2355
tsp       1933
pieces    1152
medium     956
large      428
clove      423
unit       230
slices     135
pc         102
stalks      97
small       73
whole       63
Name: count, dtype: int64


In [4]:
# Re-run the portion matcher on each non-gram ingredient and record what it picked.
# Mirrors the logic in convert_to_grams(): cup -> dl, then canonicalize, then match.

from functools import lru_cache

@lru_cache(maxsize=20000)
def portion_match_name(name: str, unit_text: str):
    if not name or not unit_text:
        return None
    u = unit_text.strip().lower()
    if u in {'cup', 'cups'}:
        u = 'dl'
    code = canonicalize_unit_text(u)
    if code is None:
        return None
    row, score = match_food_item(name, portion_df, code)
    if row is None or score == 0:
        return None  # fell back to generic (water density / 400g can / etc.)
    return str(row['food_item_en'])

non_gram['portion_match'] = [
    portion_match_name(n, u)
    for n, u in zip(non_gram['ingredient_name'].astype(str), non_gram['unit_lower'])
]
non_gram['nutrient_match'] = non_gram['matched_food_name'].astype(str).str.strip().str.lower()

non_gram['used_generic_fallback'] = non_gram['portion_match'].isna()
non_gram['has_nutrient_match'] = non_gram['matched_food_name'].notna() & (non_gram['matched_food_name'].astype(str).str.len() > 0)

print('Fell back to generic (dl=100g / tbsp=15g / can=400g / etc.):',
      f"{non_gram['used_generic_fallback'].mean():.1%}")

Fell back to generic (dl=100g / tbsp=15g / can=400g / etc.): 1.8%


In [5]:
# Compare the two matchers head-to-head: same food, different food, or undecidable.
# We say they agree if either name is a substring of the other (loose, but the
# portion table uses short labels like 'olive oil' while the nutrient DB uses
# 'olive oil, refined' or similar).

def first_word(s):
    return str(s).split(',')[0].split()[0] if s else ''

def agree(p, n):
    if not p or not n:
        return None
    p, n = p.lower(), n.lower()
    if p in n or n in p:
        return True
    # also count it as agreement if the headword (before comma) matches
    return first_word(p) == first_word(n) and len(first_word(p)) > 2

non_gram['agreement'] = [
    agree(p, n) for p, n in zip(non_gram['portion_match'], non_gram['nutrient_match'])
]

comparable = non_gram[non_gram['portion_match'].notna() & non_gram['has_nutrient_match']]
print(f'Comparable lines (both matchers returned something): {len(comparable):,}')
print(f'  Agree:    {(comparable["agreement"] == True).sum():,} ({(comparable["agreement"] == True).mean():.1%})')
print(f'  Disagree: {(comparable["agreement"] == False).sum():,} ({(comparable["agreement"] == False).mean():.1%})')

Comparable lines (both matchers returned something): 23,239
  Agree:    14,831 (63.8%)
  Disagree: 8,408 (36.2%)


In [6]:
# Top disagreements by frequency — these are the candidates for manual spot-check.

disagree = comparable[comparable['agreement'] == False].copy()

top = (disagree
       .groupby(['ingredient_name', 'unit_lower', 'portion_match', 'nutrient_match'])
       .size().reset_index(name='n')
       .sort_values('n', ascending=False)
       .head(50))

pd.set_option('display.max_colwidth', 60)
top

,ingredient_name,unit_lower,portion_match,nutrient_match,n
460,Olive oil,tbsp,"breakfast cereal with oats, fruit and oil, crüsli","oil, olive",1581
459,Olive oil,dl,"breakfast cereal with oats, fruit and oil, crüsli","oil, olive",653
1082,olive oil,tbsp,"breakfast cereal with oats, fruit and oil, crüsli","oil, olive",538
1080,olive oil,dl,"breakfast cereal with oats, fruit and oil, crüsli","oil, olive",326
465,"Olive oil, extra virgin",dl,"breakfast cereal with oats, fruit and oil, crüsli","oil, olive",215
1117,rapeseed oil,dl,"breakfast cereal with oats, fruit and oil, crüsli","oil, rapeseed",186
1039,ground cumin,tsp,"ground meat, raw","cumin seeds, ground",172
229,"Cumin, ground",tsp,"ground meat, raw","cumin seeds, ground",172
394,Ground cumin,tsp,"ground meat, raw","cumin seeds, ground",152
466,"Olive oil, extra virgin",tbsp,"breakfast cereal with oats, fruit and oil, crüsli","oil, olive",108


In [7]:
# Weight impact: how much recipe weight comes from disagreement lines?
# We compare the disagreement grams to the total non-ignored recipe weight.

non_gram['grams'] = pd.to_numeric(non_gram['grams'], errors='coerce')
debug['grams'] = pd.to_numeric(debug['grams'], errors='coerce')

total_non_ignored_g = debug.loc[~ignored, 'grams'].sum()
disagree_g = disagree['grams'].sum()
fallback_g = non_gram.loc[non_gram['used_generic_fallback'], 'grams'].sum()

print(f'Total non-ignored recipe weight:         {total_non_ignored_g/1000:>10,.0f} kg')
print(f'Weight from generic-fallback lines:      {fallback_g/1000:>10,.0f} kg  ({fallback_g/total_non_ignored_g:.1%})')
print(f'Weight from disagreement lines:          {disagree_g/1000:>10,.0f} kg  ({disagree_g/total_non_ignored_g:.1%})')
print()
print('Per-model breakdown of disagreement weight share:')
for m in MODELS:
    d_m = disagree[disagree['model'] == m]['grams'].sum()
    t_m = debug.loc[(~ignored) & (debug['model'] == m), 'grams'].sum()
    print(f'  {m:>20}: {d_m/t_m:.1%}  ({d_m/1000:,.0f} / {t_m/1000:,.0f} kg)')

Total non-ignored recipe weight:              7,364 kg
Weight from generic-fallback lines:              14 kg  (0.2%)
Weight from disagreement lines:                 494 kg  (6.7%)

Per-model breakdown of disagreement weight share:
            gpt54_mini: 5.9%  (77 / 1,298 kg)
            gpt54_nano: 5.4%  (68 / 1,252 kg)
          gemini_flash: 4.6%  (58 / 1,273 kg)
     gemini_flash_lite: 8.0%  (94 / 1,177 kg)
             grok_fast: 12.7%  (153 / 1,201 kg)
            grok4_fast: 3.8%  (45 / 1,164 kg)


In [8]:
# Optional: export the top disagreements to a CSV for manual review.
# Open in Excel/Sheets and tag each row 'wrong weight estimate' or 'fine'.

review = (disagree
          .groupby(['ingredient_name', 'unit_lower', 'portion_match', 'nutrient_match'])
          .agg(n=('grams', 'size'),
               mean_grams=('grams', 'mean'),
               total_grams=('grams', 'sum'))
          .reset_index()
          .sort_values('total_grams', ascending=False)
          .head(100))

out_path = DATA / 'portion_matcher_disagreements_top100.csv'
review.to_csv(out_path, index=False)
print(f'Wrote {len(review)} rows to {out_path}')
review.head(20)

Wrote 100 rows to /Users/sanderborge/Library/CloudStorage/OneDrive-Personlig/Master/Matseroppgave/master_oppskrifter/data/portion_matcher_disagreements_top100.csv


,ingredient_name,unit_lower,portion_match,nutrient_match,n,mean_grams,total_grams
460,Olive oil,tbsp,"breakfast cereal with oats, fruit and oil, crüsli","oil, olive",1581,14.263125,22550.00
459,Olive oil,dl,"breakfast cereal with oats, fruit and oil, crüsli","oil, olive",653,19.838591,12954.60
716,"Spring onions, chopped",medium,"cabbage, spring green","scallion, spring onion, raw",7,1600.000000,11200.00
26,"Almond milk, unsweetened",dl,milk,almond based beverage,77,131.558442,10130.00
1082,olive oil,tbsp,"breakfast cereal with oats, fruit and oil, crüsli","oil, olive",538,13.940520,7500.00
1117,rapeseed oil,dl,"breakfast cereal with oats, fruit and oil, crüsli","oil, rapeseed",186,38.032258,7074.00
1128,red bell pepper,piece,"chili pepper, red","sweet pepper, red, raw",43,163.546512,7032.50
381,"Green onions, chopped (about 30 g)",medium,"cabbage, spring green","scallion, spring onion, raw",4,1600.000000,6400.00
1140,"red bell pepper, raw",piece,"chili pepper, red","sweet pepper, red, raw",40,155.875000,6235.00
579,"Red bell pepper, diced",piece,"chili pepper, red","sweet pepper, red, raw",41,145.000000,5945.00
